In [35]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    Embedding,
    LSTM,
    Dense,
    Dropout
)

from tensorflow.keras.callbacks import EarlyStopping

In [36]:
df = pd.read_csv(
    "../datasets/processed/news.csv"
)

In [37]:
X = df["content"]

y = df["label"]

In [38]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [39]:
tokenizer = Tokenizer(
    num_words=10000
)

In [40]:
tokenizer.fit_on_texts(
    X_train
)

In [41]:
X_train_seq = tokenizer.texts_to_sequences(
    X_train
)

X_test_seq = tokenizer.texts_to_sequences(
    X_test
)

In [42]:
MAX_LENGTH = 500
X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=MAX_LENGTH
)
X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=MAX_LENGTH
)

In [43]:
model = Sequential([
    Embedding(
        input_dim=10000,
        output_dim=128
    ),

    LSTM(64),

    Dropout(0.3),

    Dense(
        32,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

In [44]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [45]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

In [46]:
history = model.fit(
    X_train_pad,
    y_train,
    validation_split=0.1,
    epochs=10,
    batch_size=64,
    callbacks=[
    early_stop
   ]
)

Epoch 1/10
506/506 ━━━━━━━━━━━━━━━━━━━━ 135s 262ms/step - accuracy: 0.9697 - loss: 0.0964 - val_accuracy: 0.9925 - val_loss: 0.0339
Epoch 2/10
506/506 ━━━━━━━━━━━━━━━━━━━━ 127s 250ms/step - accuracy: 0.9908 - loss: 0.0343 - val_accuracy: 0.9886 - val_loss: 0.0532
Epoch 3/10
506/506 ━━━━━━━━━━━━━━━━━━━━ 121s 240ms/step - accuracy: 0.9893 - loss: 0.0367 - val_accuracy: 0.9891 - val_loss: 0.0406


In [47]:
loss, accuracy = model.evaluate(
    X_test_pad,
    y_test
)
print(
    "Accuracy:",
    accuracy
)

281/281 ━━━━━━━━━━━━━━━━━━━━ 12s 41ms/step - accuracy: 0.9928 - loss: 0.0328
Accuracy: 0.9927616715431213


In [48]:
import joblib
joblib.dump(
    tokenizer,
    "../app/ml/saved_models/lstm_tokenizer.pkl"
)

['../app/ml/saved_models/lstm_tokenizer.pkl']

In [49]:
loss, accuracy = model.evaluate(
    X_test_pad,
    y_test
)

print("Accuracy:", accuracy)

281/281 ━━━━━━━━━━━━━━━━━━━━ 11s 40ms/step - accuracy: 0.9928 - loss: 0.0328
Accuracy: 0.9927616715431213


In [50]:
y_pred = model.predict(
    X_test_pad
)

281/281 ━━━━━━━━━━━━━━━━━━━━ 12s 43ms/step


In [51]:
y_pred = (
    y_pred > 0.5
).astype(int)

In [52]:
from sklearn.metrics import (
    classification_report
)

print(
    classification_report(
        y_test,
        y_pred
    )
)

              precision    recall  f1-score   support

           0       1.00      0.99      0.99      4696
           1       0.99      0.99      0.99      4284

    accuracy                           0.99      8980
   macro avg       0.99      0.99      0.99      8980
weighted avg       0.99      0.99      0.99      8980



In [53]:
model.save(
    "../app/ml/saved_models/lstm.keras"
)